In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB7, ResNet50, InceptionV3
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_efficient
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inception
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.svm import SVC
import joblib
from PIL import Image

In [2]:
DATA_DIR = "natural_disaster_dataset"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

IMG_SIZE = (224,224)
BATCH_SIZE = 16
EPOCHS_STAGE1 = 10   
EPOCHS_STAGE2 = 10
LR = 1e-4
RANDOM_STATE = 42
TOPK = 300   # XGBoost selected features; we can tune for performance
os.makedirs("ERI_outputs", exist_ok=True)


In [5]:
def collect(root):
    items=[]
    for cls in sorted(os.listdir(root)):
        p = os.path.join(root, cls)
        if os.path.isdir(p):
            for ext in ('*.jpg','*.jpeg','*.png','*.bmp'):
                items += [(f,cls) for f in glob(os.path.join(p,ext))]
    return items


train_items = collect(TRAIN_DIR)
val_items = collect(VAL_DIR)
test_items = collect(TEST_DIR)

print("Train:",len(train_items),"Val:",len(val_items),"Test:",len(test_items))
CLASSES = sorted(list({l for _,l in train_items}))
print("Classes:", CLASSES)


Train: 3322 Val: 444 Test: 662
Classes: ['cyclone', 'earthquake', 'flood', 'wildfire']


In [6]:

from PIL import Image, ImageFilter
def denoise_image(img_pil):
    
    return img_pil.filter(ImageFilter.MedianFilter(size=3))

def load_img_array(path, target=IMG_SIZE, denoise=True):
    img = Image.open(path).convert('RGB')
    if denoise:
        img = denoise_image(img)
    img = img.resize(target)
    arr = np.array(img).astype(np.float32)
    return arr

def preprocess_for_backbone(arr, backbone):
    if backbone=='efficientnet':
        return preprocess_efficient(arr.copy())
    if backbone=='resnet':
        return preprocess_resnet(arr.copy())
    if backbone=='inception':
        return preprocess_inception(arr.copy())
    return arr/255.0


In [7]:
def get_effnet():
    base = EfficientNetB7(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0],IMG_SIZE[1],3))
    out = GlobalAveragePooling2D()(base.output)
    m = Model(inputs=base.input, outputs=out); 
    for layer in m.layers: layer.trainable=False
    return m

def get_resnet():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0],IMG_SIZE[1],3))
    out = GlobalAveragePooling2D()(base.output)
    m = Model(inputs=base.input, outputs=out)
    for layer in m.layers: layer.trainable=False
    return m

def get_inception():
    base = InceptionV3(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0],IMG_SIZE[1],3))
    out = GlobalAveragePooling2D()(base.output)
    m = Model(inputs=base.input, outputs=out)
    for layer in m.layers: layer.trainable=False
    return m

eff_model = get_effnet()
res_model = get_resnet()
inc_model = get_inception()
print("Backbones loaded: eff,res,inc")


Backbones loaded: eff,res,inc


In [8]:
def extract_eri_features(items, cache_name):
    os.makedirs("ERI_cache", exist_ok=True)
    Xf = f"ERI_cache/{cache_name}_X.npy"; yf = f"ERI_cache/{cache_name}_y.npy"
    if os.path.exists(Xf) and os.path.exists(yf):
        X = np.load(Xf); y = np.load(yf, allow_pickle=True)
        return X, y
    feats=[]; labels=[]
    for path,label in tqdm(items, desc=f"Extract {cache_name}"):
        arr = load_img_array(path)
        a1 = preprocess_for_backbone(arr, 'efficientnet'); f1 = eff_model.predict(np.expand_dims(a1,0), verbose=0)[0]
        a2 = preprocess_for_backbone(arr, 'resnet'); f2 = res_model.predict(np.expand_dims(a2,0), verbose=0)[0]
        a3 = preprocess_for_backbone(arr, 'inception'); f3 = inc_model.predict(np.expand_dims(a3,0), verbose=0)[0]
        feat = np.concatenate([f1,f2,f3])
        feats.append(feat); labels.append(label)
    X = np.vstack(feats); y = np.array(labels)
    np.save(Xf, X); np.save(yf, y)
    return X,y

X_train_eri, y_train_eri = extract_eri_features(train_items, "train")
X_val_eri, y_val_eri = extract_eri_features(val_items, "val")
X_test_eri, y_test_eri = extract_eri_features(test_items, "test")
print("Feature shapes:", X_train_eri.shape, X_val_eri.shape, X_test_eri.shape)


Extract test: 100%|███████████████████████████████████████████████████████████████████████████████| 662/662 [05:52<00:00,  1.88it/s]

Feature shapes: (3322, 6656) (444, 6656) (662, 6656)


In [9]:
le = LabelEncoder(); y_train_enc = le.fit_transform(y_train_eri); y_val_enc = le.transform(y_val_eri); y_test_enc = le.transform(y_test_eri)

from tensorflow.keras import Input
inp = Input(shape=(X_train_eri.shape[1],))
x = Dense(512, activation='relu')(inp)
x = Dropout(0.5)(x)
out = Dense(len(CLASSES), activation='softmax')(x)
head1 = Model(inp,out)
head1.compile(optimizer=tf.keras.optimizers.Adam(LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
head1.fit(X_train_eri, y_train_enc, validation_data=(X_val_eri,y_val_enc), epochs=EPOCHS_STAGE1, batch_size=BATCH_SIZE)

# Evaluate Stage-1
tr_acc = head1.evaluate(X_train_eri, y_train_enc, verbose=0)[1]
val_acc = head1.evaluate(X_val_eri, y_val_enc, verbose=0)[1]
test_loss, test_acc = head1.evaluate(X_test_eri, y_test_enc, verbose=0)
y_test_pred = np.argmax(head1.predict(X_test_eri), axis=1)
pr = precision_score(y_test_enc, y_test_pred, average='macro', zero_division=0)
rc = recall_score(y_test_enc, y_test_pred, average='macro', zero_division=0)
f1 = f1_score(y_test_enc, y_test_pred, average='macro', zero_division=0)

result1 = pd.DataFrame([{
    'ensemble':'ERI-2025',
    'train_acc':round(tr_acc*100,2),
    'val_acc':round(val_acc*100,2),
    'test_acc':round(test_acc*100,2),
    'precision':round(pr*100,2),
    'recall':round(rc*100,2),
    'f1':round(f1*100,2)
}])

result1.to_csv("ERI_outputs/Table8_ERI_stage1.csv", index=False)
print("Saved ERI Stage-1"); display(result1)


Epoch 1/10
208/208 [==============================] - 1s 4ms/step - loss: 0.2375 - accuracy: 0.9220 - val_loss: 0.2056 - val_accuracy: 0.9324
Epoch 2/10
208/208 [==============================] - 1s 3ms/step - loss: 0.0794 - accuracy: 0.9750 - val_loss: 0.1910 - val_accuracy: 0.9459
Epoch 3/10
208/208 [==============================] - 1s 3ms/step - loss: 0.0470 - accuracy: 0.9859 - val_loss: 0.2062 - val_accuracy: 0.9437
Epoch 4/10
208/208 [==============================] - 1s 3ms/step - loss: 0.0247 - accuracy: 0.9928 - val_loss: 0.1937 - val_accuracy: 0.9459
Epoch 5/10
208/208 [==============================] - 1s 3ms/step - loss: 0.0153 - accuracy: 0.9961 - val_loss: 0.2187 - val_accuracy: 0.9437
Epoch 6/10
208/208 [==============================] - 1s 3ms/step - loss: 0.0157 - accuracy: 0.9946 - val_loss: 0.2030 - val_accuracy: 0.9505
Epoch 7/10
208/208 [==============================] - 1s 3ms/step - loss: 0.0131 - accuracy: 0.9967 - val_loss: 0.2612 - val_accuracy: 0.9414
Epoch 

,ensemble,train_acc,val_acc,test_acc,precision,recall,f1
0,ERI-2025,100.0,94.59,93.05,93.68,93.81,93.44


In [10]:
sm = SMOTE(random_state=RANDOM_STATE, n_jobs=-1)
X_train_sm, y_train_sm = sm.fit_resample(X_train_eri, y_train_enc)
print("Before SMOTE:", pd.Series(y_train_enc).value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_sm).value_counts().to_dict())


Before SMOTE: {1: 1013, 3: 808, 2: 805, 0: 696}
After SMOTE: {0: 1013, 1: 1013, 2: 1013, 3: 1013}


In [11]:
dtrain = xgb.DMatrix(X_train_sm, label=y_train_sm)
params = {'objective':'multi:softprob','num_class':len(CLASSES),'eta':0.1,'max_depth':6,'verbosity':0,'eval_metric':'mlogloss'}
bst = xgb.train(params, dtrain, num_boost_round=200)
joblib.dump(bst, "ERI_outputs/xgb_ERI.bst")


['ERI_outputs/xgb_ERI.bst']

In [12]:
# feature importance (gain)
imp = bst.get_score(importance_type='gain')
feat_imp = np.zeros(X_train_eri.shape[1])
for k,v in imp.items():
    idx = int(k.replace('f','')); feat_imp[idx]=v
K = min(TOPK, X_train_eri.shape[1])
topk_idx = np.argsort(feat_imp)[-K:][::-1]
np.save("ERI_outputs/topk_idx_ERI.npy", topk_idx)


In [13]:
# selected features sets
X_train_sel = X_train_sm[:, topk_idx]
X_val_sel = X_val_eri[:, topk_idx]
X_test_sel = X_test_eri[:, topk_idx]

scaler_stage2 = StandardScaler()
X_train_sel_s = scaler_stage2.fit_transform(X_train_sel)
X_val_sel_s = scaler_stage2.transform(X_val_sel)
X_test_sel_s = scaler_stage2.transform(X_test_sel)
joblib.dump(scaler_stage2, "ERI_outputs/scaler_stage2.pkl")


['ERI_outputs/scaler_stage2.pkl']

In [14]:
# Dense head on selected features
inp2 = Input(shape=(X_train_sel_s.shape[1],))
y = Dense(256, activation='relu')(inp2)
y = Dropout(0.4)(y)
out2 = Dense(len(CLASSES), activation='softmax')(y)
head2 = Model(inp2,out2)
head2.compile(optimizer=tf.keras.optimizers.Adam(LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
head2.fit(X_train_sel_s, y_train_sm, validation_data=(X_val_sel_s, y_val_enc), epochs=EPOCHS_STAGE2, batch_size=BATCH_SIZE)


Epoch 1/10
254/254 [==============================] - 1s 4ms/step - loss: 0.5259 - accuracy: 0.8073 - val_loss: 0.3324 - val_accuracy: 0.8896
Epoch 2/10
254/254 [==============================] - 1s 3ms/step - loss: 0.1608 - accuracy: 0.9492 - val_loss: 0.2813 - val_accuracy: 0.9077
Epoch 3/10
254/254 [==============================] - 1s 3ms/step - loss: 0.1130 - accuracy: 0.9647 - val_loss: 0.2592 - val_accuracy: 0.9167
Epoch 4/10
254/254 [==============================] - 1s 3ms/step - loss: 0.0941 - accuracy: 0.9694 - val_loss: 0.2445 - val_accuracy: 0.9234
Epoch 5/10
254/254 [==============================] - 1s 3ms/step - loss: 0.0777 - accuracy: 0.9746 - val_loss: 0.2410 - val_accuracy: 0.9257
Epoch 6/10
254/254 [==============================] - 1s 3ms/step - loss: 0.0660 - accuracy: 0.9798 - val_loss: 0.2300 - val_accuracy: 0.9279
Epoch 7/10
254/254 [==============================] - 1s 3ms/step - loss: 0.0564 - accuracy: 0.9842 - val_loss: 0.2274 - val_accuracy: 0.9302
Epoch 

In [15]:
tr_acc2 = head2.evaluate(X_train_sel_s, y_train_sm, verbose=0)[1]
val_acc2 = head2.evaluate(X_val_sel_s, y_val_enc, verbose=0)[1]
test_loss2, test_acc2 = head2.evaluate(X_test_sel_s, y_test_enc, verbose=0)
y_test_pred2 = np.argmax(head2.predict(X_test_sel_s), axis=1)
pr2 = precision_score(y_test_enc, y_test_pred2, average='macro', zero_division=0)
rc2 = recall_score(y_test_enc, y_test_pred2, average='macro', zero_division=0)
f12 = f1_score(y_test_enc, y_test_pred2, average='macro', zero_division=0)

result2 = pd.DataFrame([{
    'ensemble':'ERI-2025',
    'train_acc':round(tr_acc2*100,2),
    'val_acc':round(val_acc2*100,2),
    'test_acc':round(test_acc2*100,2),
    'precision':round(pr2*100,2),
    'recall':round(rc2*100,2),
    'f1':round(f12*100,2),
    'selected_features':int(K)
}])
result2.to_csv("ERI_outputs/Table10_ERI_stage2.csv", index=False)
print("Saved ERI Stage-2"); display(result2)


21/21 [==============================] - 0s 1ms/step
Saved ERI Stage-2


,ensemble,train_acc,val_acc,test_acc,precision,recall,f1,selected_features
0,ERI-2025,99.51,93.02,91.69,92.36,92.41,92.16,300


In [16]:
from sklearn.preprocessing import StandardScaler

scaler_svm = StandardScaler()
X_train_svm = scaler_svm.fit_transform(X_train_sel) 
X_test_svm = scaler_svm.transform(X_test_sel)
joblib.dump(scaler_svm, "ERI_outputs/scaler_svm.pkl")


['ERI_outputs/scaler_svm.pkl']

In [17]:
svm = SVC(kernel='rbf', C=10.0, gamma='scale', probability=True, random_state=RANDOM_STATE)
svm.fit(X_train_svm, y_train_sm)
joblib.dump(svm, "ERI_outputs/svm_ERI.pkl")

y_train_pred_svm = svm.predict(X_train_svm)
y_test_pred_svm = svm.predict(X_test_svm)
tr_acc_svm = accuracy_score(y_train_sm, y_train_pred_svm)
test_acc_svm = accuracy_score(y_test_enc, y_test_pred_svm)
pr_svm = precision_score(y_test_enc, y_test_pred_svm, average='macro', zero_division=0)
rc_svm = recall_score(y_test_enc, y_test_pred_svm, average='macro', zero_division=0)
f1_svm = f1_score(y_test_enc, y_test_pred_svm, average='macro', zero_division=0)


In [18]:
result3 = pd.DataFrame([{
    'ensemble':'ERI-2025',
    'train_acc':round(tr_acc_svm*100,2),
    'test_acc':round(test_acc_svm*100,2),
    'precision':round(pr_svm*100,2),
    'recall':round(rc_svm*100,2),
    'f1':round(f1_svm*100,2)
}])
result3.to_csv("ERI_outputs/Table11_ERI_stage3.csv", index=False)
print("Saved ERI Stage-3"); display(result3)


Saved ERI Stage-3


,ensemble,train_acc,test_acc,precision,recall,f1
0,ERI-2025,100.0,93.05,93.77,93.74,93.5


In [19]:
report = classification_report(y_test_enc, y_test_pred_svm, target_names=le.classes_, output_dict=True, zero_division=0)
rows=[]
for cls in le.classes_:
    d = report[cls]
    rows.append({'class':cls, 'precision':round(d['precision']*100,2), 'recall':round(d['recall']*100,2), 'f1-score':round(d['f1-score']*100,2), 'support':int(d['support'])})
result4 = pd.DataFrame(rows)
result4.to_csv("ERI_outputs/Table13_class_report.csv", index=False)
print("Saved ERI classification report"); display(result4)


Saved ERI classification report


,class,precision,recall,f1-score,support
0,cyclone,99.27,97.84,98.55,139
1,earthquake,95.51,84.58,89.71,201
2,flood,81.58,96.88,88.57,160
3,wildfire,98.73,95.68,97.18,162


In [20]:

probs = svm.predict_proba(X_test_svm)
auc_per_class = {}
for i,cls in enumerate(le.classes_):
    y_true_bin = (y_test_enc==i).astype(int)
    try:
        auc = roc_auc_score(y_true_bin, probs[:,i])
    except:
        auc = None
    auc_per_class[cls] = auc
pd.DataFrame([auc_per_class]).T.to_csv("ERI_outputs/auc_per_class.csv")
print("Saved AUC per class. All ERI outputs in ./ERI_outputs/")


Saved AUC per class. All ERI outputs in ./ERI_outputs/


In [21]:
from numba import cuda 
device = cuda.get_current_device()
device.reset()